In [26]:
import pandas as pd
import re

In [27]:
msg = pd.read_csv('./data/MSG.CSV', index_col=0)
print(msg.shape)
msg.head(2)

(2380798, 15)


,STND_YMD,INCS_NO,MSG_SEND_CHN_CD,MSG_SEND_CHN_NM,MSG_SEND_OFR_CHN_CD,MSG_SEND_OFR_CHN_NM,MSG_SEND_CNT,ETL_PROC_DTTM,저니,저니명,버전,발송타입,발송시간,본문,버튼
421,2024-11-13,3ee6a62a6cd178a8c9247160fc0828ae92a5cc137d7bd1...,FT,카카오 친구톡,IN24011001A,이니스프리,1,2024-11-30 04:45:31.281,IN24011001A,IN24011001A_이니스프리_신규쿠폰_베스트SC,7.0,친구톡,8-2-2024 3:01 PM,AN*************** 고객님!\n\n보유하신 쿠폰이 사라지고 있어요!\n...,"{""button"":[{""name"":""쿠폰함 보기"", ""type"":""WL"", ""url..."
422,2024-11-13,3ee6a62a6cd178a8c9247160fc0828ae92a5cc137d7bd1...,FT,카카오 친구톡,IN24011001A,이니스프리,1,2024-11-30 04:45:31.281,IN24011001A,IN24011001A_이니스프리_신규쿠폰_베스트SC,8.0,친구톡,8-9-2024 3:02 PM,AB**************** 고객님!\n\n보유하신 쿠폰이 사라지고 있어요!\...,"{""button"":[{""name"":""쿠폰함 보기"", ""type"":""WL"", ""url..."


### 메세지 indicating
* 메시지 길이
* 할인율
    * 여러개의 경우 가장 할인율이 높은 수치
    * 50% -> 50
    * 1+1 -> 50 / 2+1 -> 33
* 이모지 수
    * (하트), (우와)
* 시간압박
    * D-0
    * SOLD OUT
    * 소진 임박
* 개인화 여부
    * 00고객님, 00님

In [28]:
temp = list(set(msg['본문']))
len(temp)

135

In [24]:
msg_length_list = []
max_discount_list = []
emoji_count_list = []
time_pressure_list = []

for idx, row in msg.iterrows():
    text = row['본문']

    # 메시지 길이
    msg_length_list.append(len(row['본문']))
    
    # 할인율
    max_discount = 0  # 기본 할인율 0으로 설정
    discount_matches = re.findall(r'(\d+)%', text)
    bogo_matches = re.findall(r'(\d+)\+(\d+)', text)
        
    if discount_matches:
        max_discount = max(map(int, discount_matches))
        
    for buy, free in bogo_matches:
        discount = int(free) / (int(buy) + int(free)) * 100
        if discount > max_discount:
            max_discount = discount
    max_discount_list.append(max_discount)

    # 이모지 수
    emoji_count = text.count('(하트)') + text.count('(우와)')
    emoji_count_list.append(emoji_count)

    # 시간압박 여부
    time_pressure = int(bool(re.search(r'D-|SOLD OUT|소진 임박', text)))
    time_pressure_list.append(time_pressure)

In [25]:
msg_indicate = pd.DataFrame(
    {   
        'STND_YMD' : msg['STND_YMD'],
        'INCS_NO' : msg['INCS_NO'],
        'msg_length' : msg_length_list,
        'max_discount' : max_discount_list,
        'emoji_count' : emoji_count_list,
        'time_pressure' : time_pressure_list,

    }
)
msg_indicate.head(2)

,STND_YMD,INCS_NO,msg_length,max_discount,emoji_count,time_pressure
421,2024-11-13,3ee6a62a6cd178a8c9247160fc0828ae92a5cc137d7bd1...,115,10.0,0,0
422,2024-11-13,3ee6a62a6cd178a8c9247160fc0828ae92a5cc137d7bd1...,116,10.0,0,0


In [34]:
msg_indicate.to_csv('./data/msg_indicate.csv')